In [ ]:
Input=[('scale',StandardScaler()),('model',LinearRegression())]

pipe=Pipeline(Input)

pipe.fit(Z,y)

ypipe=pipe.predict(Z)
ypipe[0:10]

# 对前面创建的 lm = LinearRegression() 执行最小二乘拟合：在“highway-mpg vs price”散点上找一条直线，使所有样本的残差平方和最小

#这里的 X = df[['highway-mpg']]、Y = df['price']——即用“高速油耗里程”单变量预测价格，训练，训练完成后，模型内部产生两个参数
lm.fit(X, Y)
# score 是 sklearn 估计器的通用评估接口，对回归模型默认返回 R²（决定系数）
# 直观理解：价格波动中有多大的比例能被 highway-mpg 解释。
# 解读：约 49.66%——一半左右的价格差异能用油耗解释，一半解释不了（受品牌、发动机、车况等影响）。作为单变量模型，这个 R² 只算一般，说明 SLR 能力有限。
print('The R-square is: ', lm.score(X, Y))
# 这一单元是三种模型的第一个基准点。后续会依次对比：
# 模型	特征	R²（约）
# SLR（本单元）	highway-mpg	0.497
# MLR	4 个特征	0.809
# 多项式（Pipeline）	4 特征 2 阶升维	更高
# R² 逐级上升，配合 MSE 和分布图/残差图，最终支撑“多项式回归拟合效果最好”的选型结论。

# R² 是判定“模型是否合适”的必要参考，但不是充分判据：它回答“解释了多少方差”（量的问题）；“假设对不对”（形态）要靠残差图，“会不会过拟合”（泛化）要靠 MSE 对照/独立测试数据。单看 R² 定模型 = 只看分数不看错题——把它和 MSE、残差图、分布图组合使用，才是 Lab01 教给你的完整评估方法

# predict 把训练学到的公式应用到全部 201 个样本上 批量预测
Yhat=lm.predict(X)
print('The output of the first four predicted value is: ', Yhat[0:4])

# 从 sklearn.metrics（度量指标模块）导入均方误差函数 mean_squared_error。
from sklearn.metrics import mean_squared_error
# 对每个样本算“预测值 − 真实值”，平方后取平均。（这就是MSE）
# MSE 越小越好——MLR 的 MSE 比 SLR 小了一半以上，验证了 R² 上升的结论。
mse = mean_squared_error(df['price'], Yhat)
print('The mean square error of price and predicted value is: ', mse)
# 为什么有了 R² 还要 MSE
# 这两个指标互补，缺一不可：
# ① 单位与量纲不同
# R² 无量纲（0~1 的比例），适合跨模型横向比较；
# MSE 带价格的平方单位（美元²），数字巨大（10⁷ 量级），但对误差的绝对大小更敏感直观。
# ② 防止单一指标误导
# 只看 R²：加无用变量 R² 也会涨（过拟合风险被掩盖）；
# 只看 MSE：不同数据集之间没法比（量纲依赖数据本身）；
# 教材的结论方式就是“R² + MSE 组合判定”：如 "The MSE of SLR is 3.16×10⁷ while MLR has 1.2×10⁷. The MSE of MLR is much smaller." ——两个指标同向，选型结论才可靠。
# ③ 平方的含义平方有两个作用：① 消除正负误差相互抵消（预测偏高和偏低都算“错”）；② 放大大误差的惩罚——差 5000 的样本比差 1000 的样本“恶劣” 25 倍，模型会优先照顾错得离谱的预测。

lm.fit(Z, df['price'])
print('The R-square is: ', lm.score(Z, df['price']))

Y_predict_multifit = lm.predict(Z)
print('The mean square error of price and predicted value using multifit is: ', \
      mean_squared_error(df['price'], Y_predict_multifit))


# 从 sklearn.metrics 导入 R² 打分函数，和上一行的 mean_squared_error 是同一个模块的邻居——评估指标全家桶里的两个成员。
from sklearn.metrics import r2_score
# x = df['highway-mpg']              # 自变量：高速油耗里程（201 个值）
# y = df['price']                    # 因变量：真实价格（201 个值）
# f = np.polyfit(x, y, 3)            # 用 3 次多项式拟合，返回 4 个系数
# p = np.poly1d(f)                   # 打包成"多项式函数"对象 p
# y 是什么
# 真实价格数组，来自数据集 price 列，形状 (201,)。
# 在 r2_score(y, p(x)) 中充当基准答案（y_true）——模型要预测的对象。
# p(x) 是什么
# p 是 np.poly1d 打包出来的3 次多项式函数

# p(x) 表示把全部 201 个 highway-mpg 值代入这个多项式，得到 201 个预测价格——即模型的预测值数组 
# 先用 p(x) 算出 201 个预测价格；
# 再把“真实 vs 预测”交给 r2_score，算出这个 3 次多项式模型解释了多少价格方差；
# 结果存入 r_squared，第 59 行打印，输出约为 0.6744。
r_squared = r2_score(y, p(x))
print('The R-square value is: ', r_squared)

# 为什么必须用 r2_score 而不是 .score()
# 呼应你上一个问题：p 是 np.poly1d 对象，不是 sklearn 估计器，没有 .score() 方法。所以只能：
# 自己先算预测值 p(x)；
# 再调用独立函数 r2_score(y, 预测值) 打分。
# 核心原因：模型对象是谁家的，决定了能用哪种方式
# 一句话版：lm.score() 是“模型自带的打分器”，只有 sklearn 家的模型才有；p 是 NumPy 家的多项式，它没有这个方法，只能用独立的 r2_score 函数“从外面”给它打分。
# 前面（SLR/MLR/PIPELINE）：sklearn 估计器
# lm = LinearRegression()      # sklearn 家的模型
# sklearn 规定：所有估计器都必须实现统一的接口——fit / predict / score。所以 lm 天生自带 score 方法，且内部会自动帮你调 predict：
# 这里（多项式）：NumPy 的对象
# f = np.polyfit(x, y, 3)   # 返回系数数组
# p = np.poly1d(f)          # NumPy 家的多项式函数
# p 只是一个“会算多项式值的数学对象”，不存在 fit/predict/score 这套接口。你写 p.score(...) 会直接报错

# p(x)相当于yhat
mean_squared_error(df['price'], p(x))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline


# 生成从 1 开始、到 99 结束（不含 100）、步长为 1 的等差数组
# 结果是一个一维数组：[1, 2, 3, ..., 99]，共 99 个元素
# 把一维数组重塑成二维列向量，-1 表示行数自动推算，1 表示每行 1 个特征
# 为什么要 reshape？
# 这是 scikit-learn 的硬性要求：lm.predict() 期望输入是 二维结构 (n_samples, n_features)。如果直接传一维数组，会报经典错误
new_input=np.arange(1, 100, 1).reshape(-1, 1)
# 拟合
lm.fit(X, Y)
lm

# 预测价格
yhat=lm.predict(new_input)
yhat[0:5]

# 画出预测线
plt.plot(new_input, yhat)
plt.show()


